a two layer, 768 neuron LSTM got us to sub-3 loss with ~18 perplexity. whilst this is *decent*, sequence generation examples showed that the LSTM network was not able to sufficiently generalise context. for example, if a male character was given (for e.g "Once upon a time, there was a boy named Lucas"), the personal pronoun following would often be wrong ("she"). this may be due to many reasons - perhaps the name 'lucas' does not come up in the training data enough to generalise that this is a male name, or that the distance between the noun and pronoun is sufficiently large enough that the LSTM does not diffuse this information through the cell state well enough.

modern llms; including ChatGPT and Claude do not (unsurprisingly) use LSTM networks. rather, they use an architecture called *transformers*. **transformers** are *not* a recurrent neural network, and can process the supplied token sequence in parallel during training. autoregressive generation still produces new tokens one at a time. the maximum sequence length available to the model is called the *context window*.

In [5]:
import sys
from pathlib import Path
cwd = Path.cwd()
ROOT = next(p for p in (cwd, *cwd.parents) if (p / "src").is_dir())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

i attempt to manually implement a transformer model (there is a pre-baked transformer model in `torch`). a transformer has two key components:
- `self-attention`: with the $Q$ (query), $K$ (key) and $V$ (value) matrices produced by learned linear layers. $Q$ represents what we are looking for, $K$ represents what the current token is and $V$ represents what info to pass on.
- `feed-forward`: gets information from the self-attention layer and feeds forward.

there is multi-head attention which uses several attention heads in parallel, each with its own learned projections, but we omit implementing this for our homebrew model. given an input matrix $X$ with one token per row (omitting the batch dimension), we define
$$
\begin{align*}
    Q &= XW_Q^T + b_Q \\
    K &= XW_K^T + b_K \\
    V &= XW_V^T + b_V
\end{align*}
$$

the weights follow the same convention as `nn.Linear`, with biases broadcast over token positions. then, the self attention layer passes on
$$
\text{SelfAttention}(X) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V
$$
where $d_k$ is the query and key dimension, equal to the embedding dimension in our single-head implementation. the softmax runs across key positions for each query. the causal mask has $M_{ij}=0$ when $j \leq i$ and $M_{ij}=-\infty$ otherwise, so a token cannot attend to future tokens.

our implementation applies layer normalisation before each sublayer. the original input is then added back onto the attention output. this is called the "residual connection".
$$
X_{interim} = X + \text{SelfAttention}(\text{LayerNorm}_1(X))
$$
the feed forward layer applies a nonlinear activation between its two linear layers. here we use gelu, so for an input $Z$
$$
\text{FeedForward}(Z) = \text{gelu}(ZW_1^T + b_1)W_2^T + b_2
$$
then, the output of the transformer block is
$$
X_{output} = X_{interim} + \text{FeedForward}(\text{LayerNorm}_2(X_{interim}))
$$

In [139]:
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import math

from torch.utils.data import DataLoader

plt.rcParams["font.family"] = "Courier New"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from src.data.dataset import LMDataset

cuda


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embedding_dim: int):
        super().__init__()

        # [embedding_dim, embedding_dim]
        self.q_proj = nn.Linear(embedding_dim, embedding_dim)
        self.k_proj = nn.Linear(embedding_dim, embedding_dim)
        self.v_proj = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x: torch.Tensor):
        # number of batches, number of tokens, dimension embedding
        _, T, d_model = x.shape

        # get the query, key and value matrices
        # [B, T, embedding_dim] * [embedding_dim, embedding_dim]
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        # first dimension is batches
        # so we transpose the last two dimensions
        scores = Q @ K.transpose(-1, -2)
        scores = scores / math.sqrt(d_model)

        # we need to create the lower triangle, such that past tokens do not know future tokens
        mask = torch.tril(torch.ones(T, T, device=x.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        # we softmax by the tokens, such that each token's attention weights are normalised
        weights = torch.softmax(scores, dim=-1)

        return weights @ V

class FeedForward(nn.Module):
    def __init__(self, embedding_dim: int, forward_dim: int):
        super().__init__()

        # w_2 * gelu(w_1x + b_1) + b_2
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, forward_dim),
            nn.GELU(),
            nn.Linear(forward_dim, embedding_dim)
        )

    def forward(self, x: torch.tensor):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim: int, forward_dim: int):
        """
        dimension of embedding
        """
        super().__init__()

        self.lnorm_one = nn.LayerNorm(embedding_dim)
        self.self_attention = SelfAttention(embedding_dim)
        self.lnorm_two = nn.LayerNorm(embedding_dim)
        self.feed_forward = FeedForward(embedding_dim, forward_dim)

    def forward(self, x: torch.Tensor):
       x = x + self.self_attention(self.lnorm_one(x))
       x = x + self.feed_forward(self.lnorm_two(x))
       return x

the above is a fully implemented transformer block (without multi-head attention). we can then wrap the transformer block into a decoder network such that there is a final linear layer which outputs logits. we also add a positional embedding which gives context to the model on the *position* of the token in a sequence.

#### positional embeddings

rnns (and by extension, lstms) get information about token order through recurrence. since they have recurrent states $h_{t-1}$ and $c_{t-1}$ getting passed through to timestep $t$, their states depend on the order of preceding tokens. self-attention alone has no explicit position information. a causal mask limits which tokens can be seen, but we still add positional embeddings to represent where each token sits in the sequence. it's easy to understand why this is important `gloria ate the biscuit`, `biscuit ate the gloria`, etc. all have different meanings. how do we deal for this?

**positional embeddings** are an extra embedding which give the unique position of a token $p = 0, 1, 2, \dots, N-1$ its own vector $\mathbf{p}_p$, of the same length as the token embedding dimension. here $N$ is the maximum sequence length. this means that the embedding of some input token $x$ at position $p$ becomes

$$ \mathbf{x}_p = \mathbf{e}_p + \mathbf{p}_p$$

where $\mathbf{e}_p$ is the token embedding of the token at position $p$.

In [50]:
class Decoder(nn.Module):
    def __init__(self, embedding_dim: int, forward_dim: int, vocab_dim: int, max_seq_len: int, n_layers: int):
        super().__init__()

        self.max_seq_len = max_seq_len

        self.embedding = nn.Embedding(vocab_dim, embedding_dim)
        self.pos_embedding = nn.Embedding(max_seq_len, embedding_dim)
        self.transformers = nn.ModuleList([TransformerBlock(embedding_dim, forward_dim) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(embedding_dim)
        self.lm = nn.Linear(embedding_dim, vocab_dim)

    def forward(self, x):
        _, T = x.shape

        assert T <= self.max_seq_len

        emb = self.embedding(x)
        pos = torch.arange(T, device=x.device)
        pos_emb = self.pos_embedding(pos)

        x = emb + pos_emb

        for transformer in self.transformers:
            x = transformer(x)

        x = self.final_norm(x)
        logits = self.lm(x)
        return logits

let's try to train this model. we train on the tinystories dataset. we try going for 16 layers, to roughly match our two layer lstm's parameter count.

In [ ]:
from transformers import PreTrainedTokenizerFast

tokeniser = PreTrainedTokenizerFast.from_pretrained('vuiseng9/bpe-10.0k-tinystories')
tr_data = LMDataset(r'C:/data/tinystories/processed/train.bin', seq_len=128)
tst_data = LMDataset(r'C:/data/tinystories/processed/test.bin', seq_len=128)

# data loader, pin memory into cpu for fast transfer to gpu
tr_loader = DataLoader(tr_data, batch_size=64, shuffle=True, pin_memory=True, num_workers=0)

# model details
model = Decoder(256, 768, tokeniser.vocab_size, 128, 16).to(device)
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3, fused=True)
scaler = torch.amp.GradScaler("cuda")

In [153]:
from collections import deque

loss_hist = []

model.train()
total_batches = len(tr_loader)
recent_losses = deque(maxlen=500)

for epoch in range(3):
    for batch_idx, (X_batch, y_batch) in enumerate(tr_loader, start=1):
        X_batch = X_batch.to(device, dtype=torch.long, non_blocking=True)
        y_batch = y_batch.to(device, dtype=torch.long, non_blocking=True)

        optimiser.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(X_batch)
            loss = loss_fn(logits.reshape(-1, tokeniser.vocab_size), y_batch.reshape(-1))

        scaler.scale(loss).backward()
        scaler.step(optimiser)
        scaler.update()

        recent_losses.append(loss.detach().item())
        recent_loss = sum(recent_losses) / len(recent_losses)

        loss_hist.append({ 'epoch': epoch, 'i': batch_idx, 'loss': loss.detach().item(), 'recent_loss': recent_loss })

        if batch_idx % 1000 == 0 or batch_idx == total_batches:
            progress = 100 * batch_idx / total_batches

            print(
                f"batch {batch_idx}/{total_batches} ({progress:.1f}%) :: "
                f"recent loss: {recent_loss:.4f}",
                flush=True,
            )

batch 1000/56759 (1.8%) :: recent loss: 8.2188
batch 2000/56759 (3.5%) :: recent loss: 7.6687
batch 3000/56759 (5.3%) :: recent loss: 6.9426
batch 4000/56759 (7.0%) :: recent loss: 6.4555
batch 5000/56759 (8.8%) :: recent loss: 6.1704
batch 6000/56759 (10.6%) :: recent loss: 5.9770
batch 7000/56759 (12.3%) :: recent loss: 5.8404
batch 8000/56759 (14.1%) :: recent loss: 5.7266
batch 9000/56759 (15.9%) :: recent loss: 5.6438
batch 10000/56759 (17.6%) :: recent loss: 5.5793
batch 11000/56759 (19.4%) :: recent loss: 5.5230
batch 12000/56759 (21.1%) :: recent loss: 5.4741
batch 13000/56759 (22.9%) :: recent loss: 5.4336
batch 14000/56759 (24.7%) :: recent loss: 5.3959
batch 15000/56759 (26.4%) :: recent loss: 5.3675
batch 16000/56759 (28.2%) :: recent loss: 5.3396
batch 17000/56759 (30.0%) :: recent loss: 5.3140
batch 18000/56759 (31.7%) :: recent loss: 5.2931
batch 19000/56759 (33.5%) :: recent loss: 5.2753
batch 20000/56759 (35.2%) :: recent loss: 5.2534
batch 21000/56759 (37.0%) :: recen

the model learns but not well, reaching a recent training loss of `~4.52` after 3 epochs, or a perplexity of $e^{4.52} \approx 92$. this is much higher than the lstm's reported `~18`, though a fair comparison needs the same tokeniser, data split and evaluation method.

single-head attention may limit the relationships learnt within each layer, whilst 16 layers at width 256 may not be an effective use of the parameter budget. the fixed learning rate of `3e-4`, without warmup or decay, could also need tuning. these are possible explanations, not confirmed causes. the 128-token context also limits story recall, but would not explain the gap if both models had the same effective context.

the loss is still decreasing, so learning has not stopped. i'd first check the tokeniser against the training data, try overfitting a small batch, and track validation loss. we never evaluate `tst_data` here, so the training loss alone cannot tell us how well the model generalises.